In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully


In [3]:
import pandas as pd

In [4]:
movies_df=pd.read_csv("preprocessed_movies.csv")

In [5]:
movies_df.head()

,movie_id,title,tags
0,19995,Avatar,22nd century paraplegic marine dispatched moon...
1,285,Pirates of the Caribbean: At World's End,captain barbossa long believed dead come back ...
2,206647,Spectre,cryptic message bonds past sends trail uncover...
3,49026,The Dark Knight Rises,following death district attorney harvey dent ...
4,49529,John Carter,john carter warweary former military captain w...


In [7]:
embeddings=model.encode(
    movies_df['tags'].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/151 [00:00<?, ?it/s]

In [8]:
import faiss
import numpy as np

In [9]:
embeddings=np.array(embeddings).astype('float32')
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [10]:
def recommend(movie_title, top_n=5):
    #we will find movie index
    movie_idx = movies_df[
        movies_df['title']==movie_title
    ].index[0]

    #find embedding
    query_vector=embeddings[movie_idx].reshape(1,-1)

    distances, indices = index.search(
        query_vector,
        top_n+1
    )
    recommendations=[]
    for i in indices[0][1:]:
        recommendations.append(
            movies_df.iloc[i]['title']
        )
    return recommendations

In [13]:
recommend('Batman')

['Batman',
 'Batman v Superman: Dawn of Justice',
 'The Dark Knight',
 'Batman Returns',
 'Batman Begins']